# Plant Doctor — Part 3: From One Neuron to a Neural Network

*A follow-up to **Part 2 — Training, Testing & Evaluating a Model**.*

In Part 2 you trained and evaluated a **single neuron** — and saw that a single sigmoid neuron is
exactly **logistic regression**, able to draw only **one straight-line boundary**. That's the essential
building block of deep learning, but on its own it isn't yet "deep."

Part 3 makes the leap:
1. See the **same model in three lines of scikit-learn** — how you'd really train it.
2. Meet a problem one straight line **cannot** solve: plant **tolerance**.
3. **Add a hidden layer** and watch the network break through — the jump from logistic regression to a
   real (if small) **neural network**.
4. Place it all on the ladder from **one neuron → deep learning**.

> **Run the cells in order.** The first cell rebuilds the Part 2 plant dataset and model, so Part 3 is
> **self-contained** — you can run it on its own.


In [ ]:
# --- Quick recap: rebuild the Part 2 dataset + model so Part 3 stands alone ---
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# same 200-sample plant dataset + 70/30 split as Part 2
N = 200
snp, path, leaf, root = (np.random.rand(N) for _ in range(4))
X = np.column_stack([snp, path, leaf, root])
score = 1.2*snp - 2.2*path + 1.3*leaf + 1.0*root + np.random.normal(0, 0.35, N)
y = (score > 0).astype(int)                       # 1 = Healthy, 0 = Sick
perm = np.random.permutation(N); X, y = X[perm], y[perm]
split = int(0.70 * N)
X_train, X_test, y_train, y_test = X[:split], X[split:], y[:split], y[split:]

# retrain the Part 2 hand-coded single neuron (stochastic gradient descent, 60 epochs)
w, b = np.zeros(4), 0.0
for _ in range(60):
    for i in range(len(X_train)):
        p = sigmoid(np.dot(w, X_train[i]) + b)
        err = y_train[i] - p                      # (True - sigma(z))
        w += 0.05 * err * X_train[i]; b += 0.05 * err

hand_acc = np.mean((sigmoid(X_test @ w + b) >= 0.5).astype(int) == y_test)
print(f"Recap ready. Part 2's hand-coded single-neuron test accuracy: {hand_acc:.3f}")


## The same single neuron in three lines of scikit-learn

You hand-coded the whole training loop in Part 2 **so you could see exactly how a neuron learns**.
In real work you wouldn't — you'd hand it to a library. Here is the *identical* model — a single
sigmoid neuron, i.e. **logistic regression** — trained on the same plant data in three lines:


In [ ]:
from sklearn.linear_model import LogisticRegression

# C=1e6 switches OFF scikit-learn's built-in L2 regularization so this matches our
# unregularized hand-coded neuron. (Leave C at its default and it regularizes - the idea
# from Part 2's regularization aside - scoring a bit lower here, ~0.80.)
clf = LogisticRegression(C=1e6).fit(X_train, y_train)
print("scikit-learn single-neuron test accuracy:", round(clf.score(X_test, y_test), 3))
print(f"Hand-coded version scored {hand_acc:.3f} - same model, essentially the same result.")


Three lines replace the whole hand-written loop. That's the point of building it by hand first:
once you understand what's happening under the hood, the library is just a shortcut — not a black box.
And as the capstone shows next, going from this single neuron to a **multi-layer network** is barely
any more code.


---
# Capstone — From one neuron to a real network (adding a hidden layer)

Time for an honest confession about everything so far. The model you trained in Parts 1 and 2 is
**a single neuron** — and a single sigmoid neuron is *exactly* **logistic regression**. It can only
draw **one straight-line boundary** between Healthy and Sick. That's the essential building block of
deep learning, but on its own it is **not yet "deep."**

**Deep learning** means stacking many neurons into **layers**, so the network can bend and combine
boundaries into shapes a single line never could. To *see* why that matters, we need a problem one
straight line genuinely **cannot** solve — and, conveniently, plant biology hands us one:
**tolerance**.


## A problem one straight line can't solve: *tolerance*

Consider two measurements: **pathogen load** and **plant vigor**. A *tolerant* line stays **Healthy**
when its vigor roughly **keeps pace** with pathogen load. It turns **Sick** at both extremes:

- pathogen load runs **far ahead** of vigor → the plant is **overwhelmed**;
- vigor **greatly exceeds** pathogen load → the plant paid a **metabolic cost** for defenses it never needed.

So the Healthy region is a **diagonal band** through the middle — and no single straight line can fence
a band off from the two Sick corners on either side. Run the cell and look: could you separate green
from red with **one** straight line?


In [ ]:
from sklearn.model_selection import train_test_split

rng = np.random.RandomState(7)
n_tol = 400
pathogen = rng.rand(n_tol)     # feature 1: pathogen load (0-1)
vigor    = rng.rand(n_tol)     # feature 2: plant vigor (0-1)
X_tol = np.column_stack([pathogen, vigor])

# Healthy when vigor keeps pace with pathogen load -> a diagonal BAND (non-linear!)
balanced = (np.abs(vigor - pathogen) < 0.25)
y_tol = balanced.astype(int)
flip = rng.rand(n_tol) < 0.05                 # 5% label noise (nature is messy)
y_tol = np.where(flip, 1 - y_tol, y_tol)

Xtr_t, Xte_t, ytr_t, yte_t = train_test_split(X_tol, y_tol, test_size=0.3, random_state=0)

plt.figure(figsize=(5.6, 5))
plt.scatter(pathogen[y_tol==1], vigor[y_tol==1], s=14, c='#1a7f37', label='Healthy')
plt.scatter(pathogen[y_tol==0], vigor[y_tol==0], s=14, c='#c0392b', label='Sick')
plt.xlabel('Pathogen load'); plt.ylabel('Plant vigor')
plt.title('Tolerance: Healthy = a diagonal BAND\n(one straight line can never separate green from red)')
plt.legend(); plt.tight_layout(); plt.show()
print(f'{n_tol} plants  |  Healthy: {(y_tol==1).sum()}   Sick: {(y_tol==0).sum()}')


## The single neuron hits a ceiling

First we train the model you already know — a single neuron (logistic regression) — on this data.
Watch the accuracy: it stalls barely above a coin flip, and **more epochs won't rescue it**. One
straight boundary simply cannot carve out a band. This is a *structural* limit of the model, not a
training problem.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# A single sigmoid neuron == logistic regression (the exact model from Parts 1-2).
single_neuron = make_pipeline(StandardScaler(), LogisticRegression())
single_neuron.fit(Xtr_t, ytr_t)
print(f'Single neuron   -> test accuracy: {single_neuron.score(Xte_t, yte_t):.3f}')
print('Stuck near the majority-guess baseline: one straight line cannot fence off a band,')
print('no matter how long you train it.')


## The change, visualized: adding a hidden layer

Before we run it, here's the structural change in one picture — and a point worth being precise about.

On the **left** is the model you've used all along. The two **boxes** (Pathogen, Vigor) are the
**inputs** — they just *hold the two measurement values*; they are **not** neurons and compute nothing.
They both feed the **single neuron** (the `σ` circle), which reads *both* inputs, computes
`w₁·Pathogen + w₂·Vigor + b`, and squashes it with the sigmoid. So the left model is **one neuron
taking two inputs** — *not* two neurons. (A common point of confusion: input nodes are often drawn as
circles too, which makes them look like neurons — here they're boxes to keep it clear.)

On the **right**, a **hidden layer of neurons** (circles) sits between the inputs and the output.
Every input connects to every hidden neuron, and those feed the output neuron. Signal flows left →
right — that pass is **forward propagation**. Each hidden neuron learns its own simple boundary, and
the output neuron **combines** them into the curved region a single neuron can't draw.


In [ ]:
from matplotlib.patches import Circle, FancyBboxPatch

# Inputs are drawn as BOXES (they just hold a measurement); neurons are CIRCLES (they compute).
def _neuron(ax, x, yv, txt='', face='white', ec='#2E5B8A'):
    ax.add_patch(Circle((x, yv), 0.28, facecolor=face, edgecolor=ec, lw=2.2, zorder=3))
    if txt:
        ax.text(x, yv, txt, ha='center', va='center', fontsize=11, zorder=4)

def _inbox(ax, x, yv, txt):
    ax.add_patch(FancyBboxPatch((x-0.5, yv-0.2), 1.0, 0.4, boxstyle='round,pad=0.02',
                                facecolor='#eef1f4', edgecolor='#8a94a0', lw=1.6, zorder=3))
    ax.text(x, yv, txt, ha='center', va='center', fontsize=8, zorder=4)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))

# ================= LEFT: ONE neuron taking TWO inputs =================
ax = axes[0]; ax.set_xlim(0, 4.2); ax.set_ylim(0, 4); ax.axis('off')
for iy, lbl in zip([2.5, 1.3], ['Pathogen', 'Vigor']):
    _inbox(ax, 0.85, iy, lbl)
    ax.annotate('', xy=(2.78, 1.9), xytext=(1.42, iy),
                arrowprops=dict(arrowstyle='-|>', color='#888', lw=1.4))
_neuron(ax, 3.1, 1.9, 'σ', face='#dbe7f3')
ax.annotate('', xy=(3.95, 1.9), xytext=(3.42, 1.9),
            arrowprops=dict(arrowstyle='-|>', color='#888', lw=1.4))
ax.text(3.1, 3.15, '1 neuron', ha='center', fontsize=9, color='#2E5B8A', fontweight='bold')
ax.text(0.85, 3.15, '2 inputs (boxes)', ha='center', fontsize=8, color='#777')
ax.text(3.98, 1.9, 'Healthy /\nSick', ha='left', va='center', fontsize=7.5, color='#444')
ax.set_title('Single neuron (logistic regression)\n2 inputs  →  1 neuron  =  ONE straight boundary',
             fontsize=10, color='#1F3B57')

# ================= RIGHT: a hidden LAYER of neurons =================
ax = axes[1]; ax.set_xlim(0, 5.4); ax.set_ylim(0, 4); ax.axis('off')
in_ys = [2.5, 1.3]; hid_ys = [3.45, 2.72, 2.0, 1.28, 0.55]
for iy, lbl in zip(in_ys, ['Pathogen', 'Vigor']):
    _inbox(ax, 0.72, iy, lbl)
    for hy in hid_ys:
        ax.annotate('', xy=(2.62, hy), xytext=(1.24, iy),
                    arrowprops=dict(arrowstyle='-', color='#d9d9d9', lw=0.7))
for hy in hid_ys:
    _neuron(ax, 2.85, hy, ec='#C9971F')
    ax.annotate('', xy=(4.28, 1.9), xytext=(3.12, hy),
                arrowprops=dict(arrowstyle='-', color='#d9d9d9', lw=0.7))
_neuron(ax, 4.55, 1.9, 'σ', face='#dbe7f3')
ax.text(0.72, 3.6, '2 inputs', ha='center', fontsize=8, color='#777')
ax.text(2.85, 3.95, 'hidden layer of neurons', ha='center', fontsize=8.5, color='#C9971F', fontweight='bold')
ax.text(4.55, 3.15, 'output\nneuron', ha='center', fontsize=8, color='#2E5B8A')
ax.annotate('forward propagation  →', xy=(4.9, 0.15), xytext=(0.3, 0.15),
            arrowprops=dict(arrowstyle='-|>', color='#2E5B8A', lw=1.3),
            fontsize=8.5, color='#2E5B8A', va='center')
ax.set_title('One hidden layer (MLP)\n2 inputs  →  many neurons  →  1 output  =  a CURVED boundary',
             fontsize=10, color='#1F3B57')

plt.tight_layout(); plt.show()
print('Legend:  BOX = an input measurement (not a neuron)   |   CIRCLE = a neuron (computes).')
print('(5 hidden neurons drawn for clarity; the model below actually uses 16.)')


## Add ONE hidden layer, and watch it break through

> **🎯 Predict first.** The single neuron scored **0.60** on this tolerance data. Before running the
> next cell, write down your guess for what **one hidden layer** will score: *no change (~0.60)*,
> *a little better (~0.75)*, or *a lot better (~0.90)*? Then run it and see how close you were.

Now the payoff. We keep the **same data**, but insert a **hidden layer of 16 neurons** between the
inputs and the output. Each hidden neuron learns its own boundary; the output neuron then *combines*
them — which lets the network draw the curved, band-shaped region the single neuron couldn't.

We use scikit-learn's `MLPClassifier` (MLP = *multi-layer perceptron*, the classic name for this kind
of network). Two settings reflect standard real-world practice, not magic:

- **`StandardScaler`** — networks train far better when features are on a common scale.
- **`activation='relu'`** — ReLU learns much faster than stacked sigmoids (which suffer the
  "vanishing gradient" problem).

Run it: same data, same train/test split — but the accuracy jumps, and the decision-boundary plot
shows *why*.


In [ ]:
from sklearn.neural_network import MLPClassifier

network = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(16,), activation='relu', max_iter=5000, random_state=0),
)
network.fit(Xtr_t, ytr_t)
print(f'Single neuron    -> test accuracy: {single_neuron.score(Xte_t, yte_t):.3f}')
print(f'One hidden layer -> test accuracy: {network.score(Xte_t, yte_t):.3f}  <- broke through!')

# Draw both decision boundaries side by side
xx, yy = np.meshgrid(np.linspace(0, 1, 250), np.linspace(0, 1, 250))
grid = np.c_[xx.ravel(), yy.ravel()]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, model, name in [(axes[0], single_neuron, 'Single neuron (one straight line)'),
                         (axes[1], network,       'One hidden layer (16 neurons)')]:
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='RdYlGn')
    ax.scatter(pathogen[y_tol==1], vigor[y_tol==1], s=10, c='#1a7f37')
    ax.scatter(pathogen[y_tol==0], vigor[y_tol==0], s=10, c='#c0392b')
    ax.set_title(f'{name}\ntest acc = {model.score(Xte_t, yte_t):.2f}')
    ax.set_xlabel('Pathogen load'); ax.set_ylabel('Plant vigor')
plt.tight_layout(); plt.show()


## What just happened — and what "deep" really means

The single neuron drew **one straight line** and could never fence off the diagonal band, so it was
stuck near a coin flip. Adding **one hidden layer** let the network combine several boundaries into a
**curved region** that wraps the band — same data, same features, dramatically better result. That gap
is exactly what depth buys you: **stacking neurons lets a network learn patterns a single neuron
structurally cannot** — like the tolerance interaction we flagged all the way back in Part 1.

That is the leap from **logistic regression → a neural network**. Real deep-learning models simply
push this much further: *many* hidden layers, *thousands* of neurons, and *many* input features —
which is how they learn patterns in images, genomes, or sensor data far too tangled for any straight line.

**Honest caveats** (so nobody over-learns from a toy demo):
- This dataset is **synthetic and 2-D** so we could *draw* the boundary. Real field data is noisier and
  has many more features — expect messier results.
- More layers/neurons is **not automatically better**: bigger networks can **overfit** (remember the
  train-vs-test gap and Part 2's regularization aside), and they need **more data** to train well.
- We handed off the training to a library here. Under the hood, `MLPClassifier` is running the *same*
  gradient-descent + backpropagation you built by hand in Part 2 — just across more weights.


## One neuron → deep learning: where are we on the ladder?

You've now met every rung between a single neuron and "deep learning." Here's the whole progression in
one place, so the vocabulary clicks:

| Rung | What it's called | Boundary it can draw | Where you met it |
|---|---|---|---|
| **1 neuron** | a single sigmoid unit = **logistic regression** | one straight line | Parts 1 & 2 (the game + training) |
| **+ 1 hidden layer** | a **neural network** (shallow) / MLP | bent & combined — curves | the capstone above |
| **+ many hidden layers** | a **deep** neural network = **deep learning** | highly complex, layered patterns | the real-world next step |

**So what *is* "deep learning"?** It's just a neural network with **several hidden layers stacked
between input and output** — "deep" literally means *many layers*. Each layer transforms the output of
the one before it, so the network builds understanding in stages: early layers catch simple patterns,
later layers combine those into complex ones (in a leaf image, say, early layers find edges, later ones
find lesions). A **single neuron is the building block**; **one hidden layer** already makes it a
neural network; stacking **many layers** is what earns the name **deep learning**.

Crucially, **nothing about the *learning* changes as you go deeper.** It's the *same* forward pass →
error → gradient-descent update (the delta rule / backpropagation) you built by hand in Part 2, just
repeated across more neurons and more layers. You already understand the engine — deep learning is that
same engine, scaled up.


---
### Recap — across Parts 2 & 3, you can now
- Explain **why** we hold out a test set, and split data into train/test.
- Describe an **epoch** and read a **learning curve** to judge when training has converged.
- Build a **confusion matrix** and compute **accuracy, precision, recall, and F1** by hand.
- Explain why **accuracy alone is misleading**, and how the **decision threshold** trades
  precision against recall.
- Explain the ladder from a **single neuron (logistic regression)** → a **hidden layer (neural
  network)** → **many layers (deep learning)**, and *why* depth lets a model learn non-linear
  patterns like **tolerance**.

From one neuron in a classroom game to a real (if small) neural network — trained, judged, and
understood like the real thing. 🌱
